# 07 · Synthetic Durable-Goods Panel — generator + data-preparation recipe

This notebook opens the **advanced track** of the accelerator. It builds a realistic, fully synthetic
monthly panel for a **durable-goods manufacturer** (compressors, pumps, HVAC units, control systems,
spare parts & field service) whose demand is **intermittent and lumpy** for a large share of its
product lines, and it deliberately injects the data problems that cost the most time in real
projects. Every later notebook (08–12) and every new skill runs on the table this notebook writes.

No real company data is used anywhere in the advanced track. The shape of the data — hierarchy,
grain, measures, demand archetypes and injected challenges — follows the written specification in
the wiki page **Synthetic-Data-Durable-Goods** (the *data prompt*), so the same dataset can be
regenerated by hand, by this notebook, or by an LLM given that prompt.

## Data shape

| item | value |
|------|-------|
| grain | one row per **series × month**; series = `product_line` × `region` × `channel` |
| series id | `unique_id` = `"<product_line> | <region> | <channel>"` |
| calendar | monthly (`MS`), fiscal year starting **October** (`FY_START_MONTH = 10`) |
| measures | `quantity` (units shipped), `y` = net revenue (currency units), `orders` (distinct orders), `price_per_unit = y / quantity` |
| product hierarchy | `product_family` → `product_group` → `product_line` (5 × 3 × 2 = 30 lines) |
| market hierarchy | `region` (4) × `channel` (3) |
| demand archetypes | smooth · erratic · intermittent · lumpy · constant · constant_zero (ground truth kept in `_series_master`) |

## Challenges injected (all switchable in the config cell)

1. **Zero months are absent rows** (not zero rows) — the panel has gaps that must be filled.
2. **Several data extracts live in one table** (`extract_id`) — reading the whole table interleaves
   repeated series-months, so lags and growth rates leak the current month and the backtest looks
   unjustifiably good. The prep recipe **pins one extract**.
3. **Trailing-edge thinning** — late bookings mean the last 1–3 months carry fewer series than the
   steady state; after gap-filling they look like a demand collapse.
4. **UTC timestamps as strings** (`...T00:00:00Z`) — timezone-aware parsing breaks naive comparisons.
5. **A sustained, price-led surge** in one product group and in one strategic series (price up,
   units flat or down) — the regime shift tree models cannot extrapolate.
6. **New** series (first activity inside the last 12 months) and **possibly obsolete** series
   (silent for the last 12+ months).
7. **Credit notes** (negative revenue months) and **revenue without quantity** (installation /
   freight billed separately) — `price_per_unit` must be guarded.

## Outputs

| table | content |
|-------|---------|
| `<scenario>_raw` | the messy multi-extract long table, as an ERP export would look |
| `<scenario>_series_master` | one row per series: hierarchy, true archetype and flags (synthetic-only ground truth) |
| `<scenario>_prepared` | the pinned, de-duplicated, gap-filled monthly panel consumed by notebook 08 |

## 1. Imports

In [ ]:
import os
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("✅ Core libraries imported")

## 2. Configuration

Everything about the synthetic world is a parameter. The defaults produce 360 series × 72 months
(~26k prepared rows), small enough to run every advanced-track notebook on a laptop in minutes.

In [ ]:
SCENARIO = "durable_goods_monthly"
SEED = 7

# ---- calendar ----
START_MONTH = "2019-11-01"      # first month of history
END_MONTH = "2025-11-01"        # last month available in the latest extract (1st month of a fiscal quarter)
FY_START_MONTH = 10             # fiscal year starts in October: FY26 = Oct-2025 .. Sep-2026

# ---- hierarchy (product_family -> product_group -> [product_line]) ----
PRODUCT_TREE = {
    "Compressors": {"Rotary Screw": ["RS-55", "RS-160"], "Reciprocating": ["RC-15", "RC-40"],
                    "Centrifugal": ["CF-600", "CF-1200"]},
    "Pumps": {"Centrifugal Pumps": ["CP-20", "CP-75"], "Positive Displacement": ["PD-10", "PD-30"],
              "Submersible": ["SB-5", "SB-25"]},
    "HVAC Units": {"Rooftop Units": ["RT-10", "RT-30"], "Chillers": ["CH-100", "CH-400"],
                   "Heat Pumps": ["HP-8", "HP-20"]},
    "Control Systems": {"Controllers": ["CTL-100", "CTL-300"], "Sensors": ["SN-1", "SN-2"],
                        "Software Licences": ["SW-STD", "SW-ENT"]},
    "Spare Parts & Service": {"Filter Kits": ["FK-A", "FK-B"], "Seal Kits": ["SK-A", "SK-B"],
                              "Field Service": ["FS-PM", "FS-REP"]},
}
REGIONS = {"North America": 1.00, "EMEA": 0.80, "APAC": 0.60, "LATAM": 0.30}   # level multipliers
CHANNELS = {"Direct": 1.00, "Distributor": 0.60, "OEM": 0.40}
KEY_SEP = " | "

# ---- per product line: typical units / month (Direct, North America) and list price ----
LINE_PROFILE = {
    "RS-55": (14, 24_000), "RS-160": (6, 85_000), "RC-15": (25, 9_000), "RC-40": (10, 21_000),
    "CF-600": (1.2, 420_000), "CF-1200": (0.7, 880_000),
    "CP-20": (60, 3_800), "CP-75": (22, 11_500), "PD-10": (35, 5_200), "PD-30": (12, 14_000),
    "SB-5": (80, 1_900), "SB-25": (18, 7_400),
    "RT-10": (30, 16_000), "RT-30": (9, 38_000), "CH-100": (4, 95_000), "CH-400": (1.5, 310_000),
    "HP-8": (55, 6_800), "HP-20": (20, 12_500),
    "CTL-100": (120, 2_400), "CTL-300": (40, 7_800), "SN-1": (600, 180), "SN-2": (350, 390),
    "SW-STD": (150, 600), "SW-ENT": (40, 2_100),
    "FK-A": (300, 140), "FK-B": (160, 260), "SK-A": (220, 230), "SK-B": (90, 540),
    "FS-PM": (40, 1_900), "FS-REP": (15, 3_600),
}
UNITS_PER_ORDER = {"Compressors": 1.3, "Pumps": 2.5, "HVAC Units": 1.6, "Control Systems": 6.0,
                   "Spare Parts & Service": 9.0}

# ---- archetype mix per family (probabilities) ----
ARCHETYPE_MIX = {
    "Compressors": {"smooth": 0.40, "erratic": 0.20, "lumpy": 0.25, "intermittent": 0.15},
    "Pumps": {"smooth": 0.50, "erratic": 0.20, "intermittent": 0.20, "lumpy": 0.10},
    "HVAC Units": {"smooth": 0.55, "erratic": 0.20, "lumpy": 0.15, "intermittent": 0.10},
    "Control Systems": {"smooth": 0.35, "erratic": 0.30, "lumpy": 0.20, "intermittent": 0.15},
    "Spare Parts & Service": {"intermittent": 0.40, "lumpy": 0.20, "smooth": 0.15, "erratic": 0.10,
                              "constant": 0.08, "constant_zero": 0.07},
}
GROUP_OVERRIDE = {"Centrifugal": {"lumpy": 0.70, "erratic": 0.30},          # project business
                  "Chillers": {"lumpy": 0.55, "erratic": 0.25, "smooth": 0.20},
                  "Field Service": {"smooth": 0.45, "constant": 0.30, "intermittent": 0.25}}
SEASONALITY = {"HVAC Units": ("summer", 0.45), "Pumps": ("summer", 0.15), "Compressors": ("none", 0.0),
               "Control Systems": ("none", 0.0), "Spare Parts & Service": ("summer", 0.10)}
ANNUAL_PRICE_INFLATION = 0.03
QUARTER_END_LIFT = 0.18          # Direct channel books more in fiscal quarter-end months

# ---- regime shifts ----
SURGE_GROUP = "Heat Pumps"                           # sustained demand + price step-up (regulation-driven)
SURGE_MONTHS = 10                                    # length of the ramp ending at END_MONTH
SURGE_PRICE_UPLIFT, SURGE_UNITS_UPLIFT = 0.45, 0.30
STRATEGIC_SERIES = "RS-160 | North America | Direct"   # one dominant series (large share of revenue)
STRATEGIC_LEVEL_MULT = 12.0
STRATEGIC_SURGE_MONTHS = 8                           # price-led: price x1.8, units x0.8
STRATEGIC_PRICE_UPLIFT, STRATEGIC_UNITS_CHANGE = 0.80, -0.20

# ---- life-cycle flags ----
N_NEW_SERIES, NEW_SERIES_MAX_AGE = 12, 10            # first activity within the last 10 months
N_OBSOLETE_SERIES, OBSOLETE_SILENT_MONTHS = 12, 14   # silent for the last 14 months

# ---- data-quality challenges ----
INJECT_GAPS = True                   # zero months are absent rows
INJECT_EXTRACTS = True               # several extracts in one table
EXTRACT_DATES = ["2025-09-20", "2025-10-20", "2025-11-20"]   # each covers data through its own month
TRAILING_EDGE_MISSING = [0.04, 0.09, 0.18]                  # share of series missing in the last 3 months
EXTRACT_REVISION_SD = 0.03           # earlier extracts carry slightly different (unrevised) values
INJECT_UTC_STRINGS = True
CREDIT_NOTE_RATE = 0.010             # share of positive rows turned into a credit-note month
REVENUE_WITHOUT_QTY_RATE = 0.015     # share of rows with quantity 0 but revenue > 0

# ---- IO: Lakehouse first, local parquet fallback (same convention as notebooks 01-06) ----
LAKEHOUSE_NAME = "ts_forecasting"
LOCAL_FOLDER = SCENARIO
root_path = Path.cwd().parent.parent
local_dir = root_path / "data" / LOCAL_FOLDER
local_dir.mkdir(parents=True, exist_ok=True)

RAW_TABLE = f"{SCENARIO}_raw"
MASTER_TABLE = f"{SCENARIO}_series_master"
PREPARED_TABLE = f"{SCENARIO}_prepared"

rng = np.random.default_rng(SEED)
MONTHS = pd.date_range(START_MONTH, END_MONTH, freq="MS")
N = len(MONTHS)
print(f"✅ Config ready | {N} months {MONTHS[0].date()} -> {MONTHS[-1].date()} | FY starts month {FY_START_MONTH}")
print(f"   local fallback folder: {local_dir}")


def save_table(df: pd.DataFrame, name: str) -> None:
    """Write to the Lakehouse when Spark is available, otherwise to the local parquet fallback."""
    try:
        spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true") \
            .saveAsTable(f"{LAKEHOUSE_NAME}.{name}")
        print(f"✅ Saved {len(df):,} rows to {LAKEHOUSE_NAME}.{name}")
    except Exception as e:  # no Spark (local run) or no Lakehouse permission
        path = local_dir / f"{name}.parquet"
        df.to_parquet(path, index=False)
        print(f"ℹ️ Spark unavailable ({type(e).__name__}); saved {len(df):,} rows to {path}")

## 3. Fiscal calendar helpers

Durable-goods companies rarely report on calendar years. Every later notebook keys folds, seasonality
features and quarter totals on the **fiscal** calendar, so the helpers live here and are copied
verbatim downstream.

In [ ]:
def fiscal_parts(ts: pd.Timestamp, fy_start: int = FY_START_MONTH):
    """(fiscal_year, fiscal_quarter, fiscal_month) of a month-start timestamp."""
    fm = ((ts.month - fy_start) % 12) + 1
    fy = ts.year + (1 if ts.month >= fy_start else 0)
    return fy, (fm - 1) // 3 + 1, fm


def fiscal_label(ts: pd.Timestamp) -> str:
    fy, fq, _ = fiscal_parts(ts)
    return f"FY{fy % 100:02d}-Q{fq}"


_fp = np.array([fiscal_parts(m) for m in MONTHS])
FISCAL_MONTH = _fp[:, 2]
IS_QUARTER_END = np.isin(FISCAL_MONTH, [3, 6, 9, 12]).astype(float)
print("fiscal quarter of the last month:", fiscal_label(MONTHS[-1]))

## 4. Series master — hierarchy, levels and ground-truth archetypes

Every series gets a *true* demand archetype drawn from its family mix (with group overrides for
project businesses and service contracts). The master table keeps that truth so notebook 08 can report
how well an **objective** threshold search recovers it — a check that is only possible on synthetic data.

In [ ]:
def choose(rng, probs: dict) -> str:
    keys, p = list(probs), np.array(list(probs.values()), float)
    return keys[rng.choice(len(keys), p=p / p.sum())]


rows = []
for family, groups in PRODUCT_TREE.items():
    for group, lines in groups.items():
        for line in lines:
            units_dna, list_price = LINE_PROFILE[line]
            for region, r_mult in REGIONS.items():
                for channel, c_mult in CHANNELS.items():
                    mix = GROUP_OVERRIDE.get(group, ARCHETYPE_MIX[family])
                    rows.append({
                        "unique_id": KEY_SEP.join([line, region, channel]),
                        "product_family": family, "product_group": group, "product_line": line,
                        "region": region, "channel": channel,
                        "archetype_true": choose(rng, mix),
                        "units_level": units_dna * r_mult * c_mult * rng.lognormal(0, 0.25),
                        "list_price": list_price * (1 - 0.12 * (channel == "Distributor") - 0.22 * (channel == "OEM")),
                    })
master = pd.DataFrame(rows)
# a frequent-demand archetype needs a level of a few units/month to stay frequent after rounding
low = master["units_level"] < 3.0
master.loc[low & (master["archetype_true"] == "smooth"), "archetype_true"] = "intermittent"
master.loc[low & (master["archetype_true"] == "erratic"), "archetype_true"] = "lumpy"
master.loc[master["unique_id"] == STRATEGIC_SERIES, ["archetype_true", "units_level"]] = \
    ["smooth", master.loc[master["unique_id"] == STRATEGIC_SERIES, "units_level"].iloc[0] * STRATEGIC_LEVEL_MULT]

# life-cycle flags (never the strategic series, never the surge group)
_eligible = master[(master["unique_id"] != STRATEGIC_SERIES) & (master["product_group"] != SURGE_GROUP)
                   & master["archetype_true"].isin(["smooth", "erratic", "intermittent"])]
_pick = rng.choice(_eligible.index, N_NEW_SERIES + N_OBSOLETE_SERIES, replace=False)
master["is_new"] = master.index.isin(_pick[:N_NEW_SERIES])
master["is_obsolete"] = master.index.isin(_pick[N_NEW_SERIES:])
master["is_surge_group"] = master["product_group"] == SURGE_GROUP
master["is_strategic"] = master["unique_id"] == STRATEGIC_SERIES

print(f"series: {len(master)} | archetype mix (true):")
print(master["archetype_true"].value_counts().to_string())
print(f"new: {master['is_new'].sum()} | obsolete: {master['is_obsolete'].sum()} | surge group: {master['is_surge_group'].sum()} | strategic: 1")

## 5. Demand archetype generators

Each generator returns a vector of **monthly units**. The archetypes map onto the Syntetos–Boylan
quadrants that notebook 08 will try to recover: `smooth` (frequent, stable sizes), `erratic`
(frequent, very variable sizes), `intermittent` (many zero months, stable sizes), `lumpy` (many zero
months, very variable sizes — project deals), plus the degenerate `constant` and `constant_zero`.

In [ ]:
def seasonal_factor(kind: str, amplitude: float) -> np.ndarray:
    cal_month = MONTHS.month.to_numpy()
    if kind == "summer":
        return 1.0 + amplitude * np.cos((cal_month - 7) / 12.0 * 2.0 * np.pi)
    if kind == "winter":
        return 1.0 + amplitude * np.cos((cal_month - 1) / 12.0 * 2.0 * np.pi)
    return np.ones(N)


def gen_smooth(rng, level, season, trend=0.03, cv=0.18):
    mu = level * season * (1.0 + trend) ** (np.arange(N) / 12.0)
    return np.round(rng.gamma(1.0 / cv**2, mu * cv**2))


def _occurrences(occ, size):
    """Units on an occurrence month are at least one unit; zero elsewhere."""
    return np.where(occ, np.maximum(1.0, np.round(size)), 0.0)


def gen_erratic(rng, level, season, p_zero=0.05, sigma=0.95):
    occ = rng.random(N) > p_zero
    size = level * season * rng.lognormal(-sigma**2 / 2.0, sigma, N)
    return _occurrences(occ, size)


def gen_intermittent(rng, level, season, p_occ=0.35, cv=0.30):
    occ = rng.random(N) < p_occ
    size = rng.gamma(1.0 / cv**2, (level / p_occ) * season * cv**2)   # keep the mean comparable
    return _occurrences(occ, size)


def gen_lumpy(rng, level, season, p_occ=0.28, sigma=1.10):
    occ = rng.random(N) < p_occ
    size = (level / p_occ) * season * rng.lognormal(-sigma**2 / 2.0, sigma, N)
    return _occurrences(occ, size)


def gen_constant(rng, level, season):
    return np.full(N, max(2.0, round(level))) + (rng.random(N) < 0.10)   # flat with a rare +1


def gen_constant_zero(rng, level, season):
    out = np.zeros(N)
    out[rng.choice(N, rng.integers(1, 4), replace=False)] = max(1.0, round(level))
    return out


GENERATORS = {"smooth": gen_smooth, "erratic": gen_erratic, "intermittent": gen_intermittent,
              "lumpy": gen_lumpy, "constant": gen_constant, "constant_zero": gen_constant_zero}
print("✅ generators:", list(GENERATORS))

## 6. Build the clean monthly panel

Units → orders → price → revenue, then the regime shifts and life-cycle edits. This is the *truth*
the later notebooks are trying to forecast; the messy raw table is derived from it in section 7.

In [ ]:
def ramp(n_months: int, uplift: float) -> np.ndarray:
    """1.0 everywhere, then a linear ramp to (1 + uplift) over the last n_months."""
    f = np.ones(N)
    f[-n_months:] = 1.0 + uplift * np.linspace(1.0 / n_months, 1.0, n_months)
    return f


panels = []
for r in master.itertuples(index=False):
    s_kind, s_amp = SEASONALITY[r.product_family]
    season = seasonal_factor(s_kind, s_amp)
    if r.channel == "Direct":
        season = season * (1.0 + QUARTER_END_LIFT * IS_QUARTER_END)
    units = GENERATORS[r.archetype_true](rng, r.units_level, season)
    price = r.list_price * (1.0 + ANNUAL_PRICE_INFLATION) ** (np.arange(N) / 12.0) * rng.lognormal(0, 0.04, N)

    if r.is_surge_group:                                   # sustained demand + price step-up
        units = units * ramp(SURGE_MONTHS, SURGE_UNITS_UPLIFT)
        price = price * ramp(SURGE_MONTHS, SURGE_PRICE_UPLIFT)
    if r.is_strategic:                                     # price-led surge: price up, units down
        units = units * ramp(STRATEGIC_SURGE_MONTHS, STRATEGIC_UNITS_CHANGE)
        price = price * ramp(STRATEGIC_SURGE_MONTHS, STRATEGIC_PRICE_UPLIFT)
    units = np.round(np.maximum(units, 0.0))
    if r.is_new:
        units[: N - rng.integers(3, NEW_SERIES_MAX_AGE + 1)] = 0.0
    if r.is_obsolete:
        units[N - rng.integers(OBSOLETE_SILENT_MONTHS, OBSOLETE_SILENT_MONTHS + 10):] = 0.0

    upo = UNITS_PER_ORDER[r.product_family]
    orders = np.where(units > 0, np.maximum(1, np.round(units / upo * rng.lognormal(0, 0.2, N))), 0)
    revenue = units * price
    panels.append(pd.DataFrame({"unique_id": r.unique_id, "ds": MONTHS, "quantity": units,
                                "orders": orders, "y": revenue}))

truth = pd.concat(panels, ignore_index=True)

# credit notes (negative months, never on the strategic series) and revenue booked without quantity
pos = truth.index[(truth["y"] > 0) & (truth["unique_id"] != STRATEGIC_SERIES)]
cn = rng.choice(pos, int(CREDIT_NOTE_RATE * len(pos)), replace=False)
truth.loc[cn, "y"] = -0.35 * truth.loc[cn, "y"]
truth.loc[cn, "quantity"] = 0.0
zero = truth.index[(truth["quantity"] == 0) & (truth["y"] == 0)]
rq = rng.choice(zero, int(REVENUE_WITHOUT_QTY_RATE * len(truth)), replace=False)
truth.loc[rq, "y"] = truth["unique_id"].map(master.set_index("unique_id")["list_price"]).loc[rq] * rng.uniform(0.05, 0.3, len(rq))
truth.loc[rq, "orders"] = 1.0

# fold in hierarchy + fiscal columns
truth = truth.merge(master[["unique_id", "product_family", "product_group", "product_line", "region", "channel"]],
                    on="unique_id", how="left")
tot = truth["y"].sum()
share_strat = truth.loc[truth["unique_id"] == STRATEGIC_SERIES, "y"].sum() / tot
print(f"clean panel: {truth.shape} | total revenue {tot/1e9:.2f} bn | zero-quantity months {(truth['quantity']==0).mean():.1%}")
print(f"strategic series share of revenue: {share_strat:.1%} | surge group share: "
      f"{truth.loc[truth['product_group']==SURGE_GROUP,'y'].sum()/tot:.1%}")
truth.groupby("product_family")["y"].sum().div(1e6).round(1).rename("revenue_M")

## 7. Derive the messy raw table

This is what the extract from the order-management system looks like **before** anyone has cleaned it:
only months with activity are present, three extracts are stacked in one table, the newest months
are incomplete because invoices post late, and the period column is a UTC string.

In [ ]:
raw_parts = []
end_idx = {d: MONTHS.get_loc(pd.Timestamp(d).replace(day=1)) for d in EXTRACT_DATES} if INJECT_EXTRACTS \
    else {EXTRACT_DATES[-1]: N - 1}
for extract_date, last_idx in end_idx.items():
    part = truth[truth["ds"] <= MONTHS[last_idx]].copy()
    if INJECT_GAPS:                                   # zero months are simply not exported
        part = part[(part["quantity"] != 0) | (part["y"] != 0)]
    # late bookings: the newest months of EVERY extract are incomplete
    for k, miss in enumerate(reversed(TRAILING_EDGE_MISSING)):
        m = MONTHS[last_idx - k]
        idx = part.index[part["ds"] == m]
        part = part.drop(rng.choice(idx, int(miss * len(idx)), replace=False))
    # older extracts carry unrevised values for their last two months
    if last_idx < N - 1:
        rev_idx = part.index[part["ds"] >= MONTHS[last_idx - 1]]
        part.loc[rev_idx, "y"] = part.loc[rev_idx, "y"] * rng.lognormal(0, EXTRACT_REVISION_SD, len(rev_idx))
    part["extract_id"] = pd.Timestamp(extract_date).strftime("EXT%Y%m%d")
    part["extract_ts"] = pd.Timestamp(extract_date)
    raw_parts.append(part)

raw = pd.concat(raw_parts, ignore_index=True)
raw = raw.rename(columns={"ds": "period_start_utc", "y": "net_revenue_usd", "quantity": "units_shipped",
                          "orders": "order_count", "product_line": "product_line_code"})
if INJECT_UTC_STRINGS:
    raw["period_start_utc"] = raw["period_start_utc"].dt.strftime("%Y-%m-%dT00:00:00Z")
raw = raw.sample(frac=1.0, random_state=SEED).reset_index(drop=True)      # exports are never sorted
print(f"raw table: {raw.shape} | extracts: {raw['extract_id'].nunique()} | "
      f"duplicate (series, month) keys across extracts: {raw.duplicated(['unique_id','period_start_utc']).sum():,}")
raw.head()

In [ ]:
save_table(raw, RAW_TABLE)
save_table(master, MASTER_TABLE)

## 8. Data-preparation recipe (the lessons, in order)

| step | why it matters |
|------|----------------|
| **Pin one extract** | several extracts in one table repeat every series-month; lags computed on the stacked table see the *current* month as "lag 2" and every growth rate collapses to 0 → a leaked, unjustifiably low error. Pick the latest `extract_ts` (or a Job-provided id) **before** any feature is built. |
| **Parse timestamps as UTC, then drop the timezone** | `'2025-09-01T00:00:00Z'` parses tz-aware; comparing it to a naive `pd.Timestamp` raises or silently misses the last month. |
| **De-duplicate on (series, month), keep last** | a safety net for revisions inside one extract. |
| **Fill the full monthly grid** | absent months are zeros for quantity / orders / revenue — the intermittent profile depends on them. |
| **Guard `price_per_unit`** | `revenue / 0` for credit notes and installation-only months; keep `NaN`, never `inf`. |
| **Measure trailing-edge coverage** | if the last month carries far fewer active series than the steady state, decide explicitly whether to drop it (`TRAILING_EDGE_MIN_COVERAGE`) instead of training on false zeros. |

In [ ]:
TRAILING_EDGE_MIN_COVERAGE = 0.90     # drop trailing months whose active-series count < 90% of the recent median
PIN_EXTRACT_ID = None                 # e.g. 'EXT20250920' to force one extract; None = latest extract_ts

# 1) pin one extract
latest = raw.sort_values("extract_ts")["extract_id"].iloc[-1]
use_extract = PIN_EXTRACT_ID or latest
one = raw[raw["extract_id"] == use_extract].copy()
print(f"extracts in table: {sorted(raw['extract_id'].unique())} -> using {use_extract}")

# 2) timestamps: UTC-aware parse, then naive month-start
one["ds"] = pd.to_datetime(one["period_start_utc"], utc=True).dt.tz_localize(None).dt.to_period("M").dt.to_timestamp()

# 3) canonical names + de-dup
one = one.rename(columns={"net_revenue_usd": "y", "units_shipped": "quantity", "order_count": "orders",
                          "product_line_code": "product_line"})
one = one.sort_values(["unique_id", "ds"]).drop_duplicates(["unique_id", "ds"], keep="last")

# 4) full monthly grid per series (from the global start to the extract's last month)
data_max = one["ds"].max()
grid = pd.MultiIndex.from_product([one["unique_id"].unique(), pd.date_range(MONTHS[0], data_max, freq="MS")],
                                  names=["unique_id", "ds"]).to_frame(index=False)
prepared = grid.merge(one[["unique_id", "ds", "y", "quantity", "orders"]], on=["unique_id", "ds"], how="left")
prepared[["y", "quantity", "orders"]] = prepared[["y", "quantity", "orders"]].fillna(0.0)
prepared = prepared.merge(master[["unique_id", "product_family", "product_group", "product_line", "region", "channel"]],
                          on="unique_id", how="left")

# 5) guarded price + fiscal calendar
prepared["price_per_unit"] = np.where(prepared["quantity"] > 0, prepared["y"] / prepared["quantity"].replace(0, np.nan), np.nan)
fp = np.array([fiscal_parts(d) for d in prepared["ds"]])
prepared["fiscal_year"], prepared["fiscal_quarter"], prepared["fiscal_month"] = fp[:, 0], fp[:, 1], fp[:, 2]
prepared["fiscal_quarter_label"] = prepared["ds"].map(fiscal_label)
prepared["extract_id"] = use_extract

# 6) trailing-edge coverage: active series per month vs the median of the prior 12 months
active = prepared[(prepared["quantity"] > 0) | (prepared["y"] != 0)].groupby("ds")["unique_id"].nunique()
steady = active.iloc[-15:-3].median()
coverage = (active.iloc[-6:] / steady).round(3)
print("\nactive series per month (last 6) / steady-state median:")
print(coverage.to_string())
drop_months = [m for m in coverage.index[-3:] if coverage.loc[m] < TRAILING_EDGE_MIN_COVERAGE]
if drop_months:
    print(f"⚠️ trailing-edge thinning: dropping {[m.strftime('%Y-%m') for m in drop_months]} "
          f"(coverage < {TRAILING_EDGE_MIN_COVERAGE:.0%}) -> DATA_MAX moves to {max(prepared['ds'][~prepared['ds'].isin(drop_months)]).date()}")
    prepared = prepared[~prepared["ds"].isin(drop_months)]
else:
    print("trailing edge within tolerance; keeping every month")
prepared = prepared.sort_values(["unique_id", "ds"]).reset_index(drop=True)
print(f"\nprepared panel: {prepared.shape} | series {prepared['unique_id'].nunique()} | "
      f"{prepared['ds'].min().date()} -> {prepared['ds'].max().date()}")

### Why pinning matters — the leak, measured

Compute a 2-month lag once on the **stacked** table and once on the **pinned** table. On the stacked
table the "lag" is frequently the same month from another extract, so the lag-to-actual correlation
is inflated — exactly the mechanism that produced a too-good backtest in the field.

In [ ]:
def lag_corr(frame):
    f = frame.sort_values(["unique_id", "ds"]).copy()
    f["lag2"] = f.groupby("unique_id")["y"].shift(2)
    m = f["lag2"].notna() & (f["y"] != 0)
    return np.corrcoef(f.loc[m, "y"], f.loc[m, "lag2"])[0, 1], float((f.loc[m, "lag2"] == f.loc[m, "y"]).mean())


stacked = raw.rename(columns={"net_revenue_usd": "y"}).copy()
stacked["ds"] = pd.to_datetime(stacked["period_start_utc"], utc=True).dt.tz_localize(None)
c_stacked, same_stacked = lag_corr(stacked[["unique_id", "ds", "y"]])
c_pinned, same_pinned = lag_corr(prepared[["unique_id", "ds", "y"]])
print(f"corr(y, lag2)  stacked extracts: {c_stacked:.3f} | pinned extract: {c_pinned:.3f}")
print(f"share of rows where 'lag2' equals the current value: stacked {same_stacked:.1%} vs pinned {same_pinned:.1%}")

In [ ]:
save_table(prepared, PREPARED_TABLE)

## 9. Acceptance checks and a look at the archetypes

These checks are the contract the data prompt promises: the share of zero months, the number of series
per archetype, the surge magnitude, and the dominance of the strategic series. One sample series per
archetype is plotted so the quadrants are visible before any model is fitted.

In [ ]:
chk = prepared.merge(master[["unique_id", "archetype_true", "is_surge_group", "is_strategic"]], on="unique_id")
zero_share = (chk["quantity"] == 0).mean()
by_arch = chk.groupby("archetype_true").agg(series=("unique_id", "nunique"),
                                             zero_months=("quantity", lambda s: (s == 0).mean()),
                                             revenue_M=("y", lambda s: s.sum() / 1e6)).round(3)
last6 = chk[chk["ds"] > chk["ds"].max() - pd.DateOffset(months=6)]
prior = chk[(chk["ds"] <= chk["ds"].max() - pd.DateOffset(months=12)) & (chk["ds"] > chk["ds"].max() - pd.DateOffset(months=18))]
surge_ratio = last6.loc[last6["is_surge_group"], "y"].sum() / prior.loc[prior["is_surge_group"], "y"].sum()
strat_ratio = last6.loc[last6["is_strategic"], "y"].sum() / prior.loc[prior["is_strategic"], "y"].sum()
print(f"zero-quantity share of all series-months: {zero_share:.1%}")
print(by_arch.to_string())
print(f"\nsurge group  (last 6 months vs same window a year earlier): x{surge_ratio:.2f}")
print(f"strategic series (last 6 vs a year earlier): x{strat_ratio:.2f} | share of total revenue "
      f"{chk.loc[chk['is_strategic'],'y'].sum()/chk['y'].sum():.1%}")

In [ ]:
archetypes = ["smooth", "erratic", "intermittent", "lumpy", "constant", "constant_zero"]
fig, axes = plt.subplots(len(archetypes), 1, figsize=(11, 2.2 * len(archetypes)), sharex=True)
for ax, a in zip(axes, archetypes):
    ids = master.loc[(master["archetype_true"] == a) & ~master["is_new"] & ~master["is_obsolete"], "unique_id"]
    uid = ids.iloc[0]
    s = prepared[prepared["unique_id"] == uid].set_index("ds")["quantity"]
    ax.bar(s.index, s.values, width=20, color="steelblue")
    ax.set_title(f"{a}  |  {uid}", fontsize=9, loc="left")
    ax.set_ylabel("units")
fig.suptitle("One sample series per true archetype (monthly units)", y=1.0)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 3.2))
tot_m = prepared.groupby("ds")["y"].sum() / 1e6
strat_m = prepared[prepared["unique_id"] == STRATEGIC_SERIES].groupby("ds")["y"].sum() / 1e6
surge_m = prepared[prepared["product_group"] == SURGE_GROUP].groupby("ds")["y"].sum() / 1e6
ax.plot(tot_m.index, tot_m.values, color="black", label="total revenue")
ax.plot(strat_m.index, strat_m.values, color="firebrick", label=f"strategic series ({STRATEGIC_SERIES})")
ax.plot(surge_m.index, surge_m.values, color="darkorange", label=f"surge group ({SURGE_GROUP})")
ax.set_ylabel("revenue (M)")
ax.legend(fontsize=8)
ax.set_title("Portfolio total vs the two regime-shift stories", loc="left")
fig.tight_layout()
plt.show()

## ✅ CHECK POINT with the data scientist

- The prepared table is `<scenario>_prepared`, keyed on `unique_id` × `ds`, with `y`, `quantity`,
  `orders`, `price_per_unit`, the two hierarchies and the fiscal calendar.
- `DATA_MAX` may be **earlier than the extract's last month** if trailing-edge thinning crossed the
  coverage threshold — print it and agree on it before profiling.
- The series master is synthetic ground truth. Use it to validate notebook 08's labels, never as a
  model input.

**Next:** [08 · Profiling threshold search](08-Profiling-Threshold-Search) reads `<scenario>_prepared`.